<a href="https://colab.research.google.com/github/TahaNawazkhan/Artificial-intelligence-/blob/main/lab%203%20task%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time

START = (1, 2, 3, 4, 0, 6, 7, 5, 8)
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)


def inversions(state):
    t = [x for x in state if x != 0]
    return sum(1 for i in range(len(t)) for j in range(i + 1, len(t)) if t[i] > t[j])


def is_solvable(start, goal):
    return inversions(start) % 2 == inversions(goal) % 2


def successors(state):
    """Yield (move_name, next_state) for every legal blank move."""
    b = state.index(0)
    r, c = divmod(b, 3)
    for name, dr, dc in (("Up", -1, 0), ("Down", 1, 0),
                         ("Left", 0, -1), ("Right", 0, 1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            n = nr * 3 + nc
            s = list(state)
            s[b], s[n] = s[n], s[b]
            yield name, tuple(s)


def depth_limited_search(state, goal, limit, path_states, moves, stats):
    stats["visited"] += 1                  # node examined (goal-tested)
    if state == goal:
        return True
    if limit == 0:
        return False                       # at the limit: tested, not expanded

    stats["expanded"] += 1                 # successors are generated
    for name, nxt in successors(state):
        if nxt in path_states:             # avoid cycles along the current path
            continue
        path_states.add(nxt)
        moves.append(name)
        if depth_limited_search(nxt, goal, limit - 1, path_states, moves, stats):
            return True
        moves.pop()
        path_states.remove(nxt)
    return False


def iddfs(start, goal, max_depth=31):      # 31 = longest optimal 8-puzzle solution
    total_expanded = total_visited = 0
    per_iter = []
    for limit in range(max_depth + 1):
        stats = {"visited": 0, "expanded": 0}
        moves = []
        found = depth_limited_search(start, goal, limit, {start}, moves, stats)
        total_expanded += stats["expanded"]
        total_visited += stats["visited"]
        per_iter.append((limit, stats["expanded"], stats["visited"]))
        if found:
            return limit, moves, total_expanded, total_visited, per_iter
    return None


if __name__ == "__main__":
    print("Start :", START)
    print("Goal  :", GOAL)

    if not is_solvable(START, GOAL):
        print("This start state is UNSOLVABLE - no search started.")
        raise SystemExit

    t0 = time.perf_counter()
    depth, moves, expanded, visited, per_iter = iddfs(START, GOAL)
    secs = time.perf_counter() - t0

    print("\nPer-iteration counts:")
    print(f"{'limit':>6} {'expanded':>10} {'visited':>9}")
    for limit, e, v in per_iter:
        print(f"{limit:>6} {e:>10} {v:>9}")

    print(f"\nGoal found at depth       : {depth}")
    print(f"Solution moves            : {' -> '.join(moves)}")
    print(f"Nodes expanded (all iters): {expanded}")
    print(f"Nodes visited  (all iters): {visited}")
    print(f"Time                      : {secs*1000:.2f} ms")

Start : (1, 2, 3, 4, 0, 6, 7, 5, 8)
Goal  : (1, 2, 3, 4, 5, 6, 7, 8, 0)

Per-iteration counts:
 limit   expanded   visited
     0          0         1
     1          1         5
     2          3         7

Goal found at depth       : 2
Solution moves            : Down -> Right
Nodes expanded (all iters): 4
Nodes visited  (all iters): 13
Time                      : 0.05 ms
